# 03 – Preprocessing (Member 3: Salma)

**Project:** re-implementation of the BiLSTM emotion classifier of Glenn, LaCasse & Cox (2023) on the Arabic Egyptian Tweets Dataset (AETD), 5 classes: anger, joy, sadness, love, fear.

**Input:** the FINAL split from notebook 02 – `train.csv` (4,431), `validation.csv` (950), `test.csv` (950). Columns `ID, TWEET, LABEL`. We never change the split and never remove rows.

**What this notebook does**

| Step | Source |
|---|---|
| Remove punctuation, numbers and special characters | Paper |
| Lowercase | Paper |
| Remove stop words (NLTK Arabic list, **keeping** the negations لا، لم، لن، ما، ليس) | Paper + our choice |
| Keras `Tokenizer` fitted on train only, `num_words=20000` | Paper (Table 7: 2,000,000 params / 100 dims) |
| Zero-pad / truncate to length 64 (`pre`, Keras default) | Paper |
| Unify alef forms, remove diacritics, remove tatweel | Our additions for Arabic |

**Outputs:** `preprocessing_outputs.zip` with the cleaned CSVs, padded sequences (`.npz`), the tokenizer, the vocabulary, a config file and statistics for the report.

## 1. Setup

**What:** installs `emoji` if it is missing, imports all libraries, downloads the NLTK stop-word lists and imports the Keras `Tokenizer`.

**Why:** we need `emoji` to delete emojis, NLTK for the Arabic stop words and Keras for tokenizing and padding (exactly the tools the paper used).
The old Keras `Tokenizer` was removed in **Keras 3**. If the normal import fails, we install `tf-keras` (the Keras 2 package) and import the Tokenizer from there. The Tokenizer behaves the same in both. We also create a `stats` list that collects every number we print, so we can save them all at the end.

In [6]:
import sys, subprocess, importlib, os, re, json, shutil

# Install emoji only if it is missing
try:
    import emoji
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "emoji"])
    import emoji

import numpy as np
import pandas as pd
import nltk
import tensorflow as tf

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

# Keras Tokenizer (Keras 2 API). Keras 3 removed it -> fall back to tf-keras.
try:
    from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
    TOKENIZER_SOURCE = "tensorflow.keras"
except (ImportError, AttributeError):
    tf_ver = ".".join(tf.__version__.split(".")[:2])          # e.g. "2.19"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"tf-keras=={tf_ver}.*"])
    from tf_keras.preprocessing.text import Tokenizer, tokenizer_from_json
    TOKENIZER_SOURCE = "tf_keras"

from tensorflow.keras.utils import pad_sequences

# Fixed settings agreed by the team
MAX_LEN = 64
NUM_WORDS = 20000
PADDING = "pre"
TRUNCATING = "pre"
CLASSES = ["anger", "joy", "sadness", "love", "fear"]
NEGATIONS = ["لا", "لم", "لن", "ما", "ليس", "ولا", "فلا", "وما", "ليست", "لست", "لسنا", "لستم", "لستن", "لسن", "ليسا", "ليستا", "ليسوا", "غير"]
SEED = 42

# Collect every number we print, saved later to preprocessing_stats.csv
stats = []
def add_stat(name, value):
    stats.append({"statistic": name, "value": value})

print("TensorFlow:", tf.__version__)
print("Keras     :", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "n/a")
print("Tokenizer imported from:", TOKENIZER_SOURCE)
print("emoji     :", emoji.__version__, "| nltk:", nltk.__version__)

TensorFlow: 2.21.0
Keras     : 3.13.2
Tokenizer imported from: tensorflow.keras
emoji     : 2.16.0 | nltk: 3.9.1


## 2. Load the final split

**What:** reads the three CSV files directly from our GitHub repository. If that fails (no internet / private repo), it opens an upload box so you can upload `train.csv`, `validation.csv` and `test.csv` by hand.

**Why:** everybody must work on the exact same split. The `assert` lines stop the notebook immediately if the files have the wrong number of rows or wrong columns, so we never preprocess the wrong data by mistake.

In [7]:
BASE_URL = "https://raw.githubusercontent.com/Salmaazoz22/arabic-tweet-emotion-bilstm/main/data/"
FILES = {"train": "train.csv", "validation": "validation.csv", "test": "test.csv"}

try:
    dfs = {k: pd.read_csv(BASE_URL + f, encoding="utf-8") for k, f in FILES.items()}
    print("Loaded from GitHub.")
except Exception as e:
    print("GitHub loading failed:", e)
    print("Please upload train.csv, validation.csv and test.csv")
    from google.colab import files
    files.upload()                                   # saves the files in the current folder
    dfs = {k: pd.read_csv(f, encoding="utf-8") for k, f in FILES.items()}

train_df, val_df, test_df = dfs["train"], dfs["validation"], dfs["test"]

# The split is FINAL: check it is exactly what we expect
assert train_df.shape == (4431, 3), train_df.shape
assert val_df.shape == (950, 3), val_df.shape
assert test_df.shape == (950, 3), test_df.shape
for d in (train_df, val_df, test_df):
    assert list(d.columns) == ["ID", "TWEET", "LABEL"], list(d.columns)

print("train:", train_df.shape, "| validation:", val_df.shape, "| test:", test_df.shape)
train_df.head()

Loaded from GitHub.
train: (4431, 3) | validation: (950, 3) | test: (950, 3)


,ID,TWEET,LABEL
0,8453,خايف علي قلبي يبات مجروح,fear
1,5236,sh999 ده اللي بعملو حاليا بس خايفه م اللي بعد كده,fear
2,1487,انا الحزن و عمُر المواجع ، لا غاب صاحبّي,sadness
3,7932,انا الدوله بتاعتي اسمها زمالك,joy
4,1040,واو زيكو وبيتي قمرااات ربنا يحفظهم زكي شريف IbtissamTiskat ابتسام تسكت ZakiChreifinNewYear,joy


## 3. The `clean_text` function

**What:** one function that cleans a single tweet, step by step, in this order:
1. lowercase (for the English words in the tweets),
2. remove links, `@mentions` and the `[URL]` / `[USERNAME]` placeholders,
3. remove the `#` sign but keep the hashtag word,
4. remove emojis,
5. remove Arabic diacritics (tashkeel) and tatweel `ـ`,
6. unify أ إ آ ٱ into ا,
7. reduce a character repeated 3+ times to one (`حلوووو` → `حلو`),
8. remove numbers (Western and Arabic-Indic digits),
9. replace punctuation and special characters (also Arabic ، ؛ ؟ and `_`) with a **space**, so joined words become separate words (`#يوم_حلو` → `يوم حلو`, `كويس.جدا` → `كويس جدا`),
10. collapse multiple spaces.

**Why:** the paper removes punctuation, numbers and special characters and lowercases the text. Steps 5–6 are our additions because the same Arabic word is often written with or without diacritics or with different alef forms; unifying them means the model sees *one* word instead of several. These are the same rules Rawan used for duplicate detection in notebook 02, with one difference: here punctuation and `_` become a space instead of being deleted, so two words joined by punctuation or a hashtag underscore are not glued into one unknown word. The small examples at the end show the function working.

In [8]:
URL_RE        = re.compile(r"(?:https?://\S+|www\.\S+|\[url\])", flags=re.IGNORECASE)
MENTION_RE    = re.compile(r"(?:@\w+|\[username\])", flags=re.IGNORECASE)
DIACRITICS_RE = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")
TATWEEL       = "ـ"
ALEF_RE       = re.compile(r"[أإآٱ]")
REPEATED_RE   = re.compile(r"(.)\1{2,}")
NUMBERS_RE    = re.compile(r"\d+")            # \d also matches Arabic-Indic digits ٠-٩ and ۰-۹
PUNCT_RE      = re.compile(r"[^\w\s]|_")      # anything not letter/digit/space, plus underscore

def clean_text(text):
    text = str(text).lower()                          # 1. lowercase
    text = URL_RE.sub(" ", text)                      # 2. links + [URL]
    text = MENTION_RE.sub(" ", text)                  #    @mentions + [USERNAME]
    text = text.replace("#", "")                      # 3. keep hashtag word
    text = emoji.replace_emoji(text, replace="")      # 4. emojis
    text = DIACRITICS_RE.sub("", text)                # 5. diacritics
    text = text.replace(TATWEEL, "")                  #    tatweel
    text = ALEF_RE.sub("ا", text)                     # 6. unify alef
    text = REPEATED_RE.sub(r"\1", text)               # 7. 3+ repeats -> 1
    text = NUMBERS_RE.sub("", text)                   # 8. numbers
    text = PUNCT_RE.sub(" ", text)                    # 9. punctuation / symbols -> space
    text = re.sub(r"\s+", " ", text).strip()          # 10. spaces
    return text

examples = [
    "أنااااا مبسوووط جداً 😂😂 @ahmed #يوم_حلو https://t.co/abc 2023",
    "إنـــتَ فين؟؟ [USERNAME] ده كلام كويس.جدا،ماشي ١٢٣ [URL]",
    "LOL هههههه!!! ٱلحمد لله على كل حال ❤️",
]
for t in examples:
    print("BEFORE:", t)
    print("AFTER :", clean_text(t))
    print()

BEFORE: أنااااا مبسوووط جداً 😂😂 @ahmed #يوم_حلو https://t.co/abc 2023
AFTER : انا مبسوط جدا يوم حلو

BEFORE: إنـــتَ فين؟؟ [USERNAME] ده كلام كويس.جدا،ماشي ١٢٣ [URL]
AFTER : انت فين ده كلام كويس جدا ماشي

BEFORE: LOL هههههه!!! ٱلحمد لله على كل حال ❤️
AFTER : lol ه الحمد لله على كل حال



## 4. Stop words (keeping negations)

**What:** loads the NLTK Arabic stop-word list, passes every stop word through `clean_text` (so that, for example, `إذا` becomes `اذا` and matches our cleaned tweets), then removes the five negation words لا، لم، لن، ما، ليس from the list. Finally it defines `remove_stopwords(text)`.

**Why:** the paper removes stop words. But negations change the emotion of a sentence: *"لا أحب"* / *"ما فرحتش"* is the opposite of happy/love. If we deleted them, "I love" and "I do not love" would look the same to the model, so we **keep** them. Cleaning the stop words first is necessary, otherwise many of them would never match the cleaned text.

In [9]:
raw_stop = stopwords.words("arabic")
print("NLTK Arabic stop words (original):", len(raw_stop))

# Clean every stop word the same way as the tweets; drop empties and duplicates
stop_clean = {clean_text(w) for w in raw_stop} - {""}
print("After clean_text (unique):        ", len(stop_clean))

# Keep the negation words in the text -> remove them from the stop list
STOP_WORDS = stop_clean - set(NEGATIONS)
print("After removing the negation words:", len(STOP_WORDS))

for w in NEGATIONS:
    assert w not in STOP_WORDS, w
print("Negations NOT in the stop list:", NEGATIONS)

add_stat("stopwords_nltk_original", len(raw_stop))
add_stat("stopwords_after_clean_text", len(stop_clean))
add_stat("stopwords_final", len(STOP_WORDS))

def remove_stopwords(text):
    return " ".join(w for w in text.split() if w not in STOP_WORDS)

print(remove_stopwords("انا لا احب هذا الكلام من غير سبب"))

NLTK Arabic stop words (original): 754
After clean_text (unique):         628
After removing the negation words: 610
Negations NOT in the stop list: ['لا', 'لم', 'لن', 'ما', 'ليس', 'ولا', 'فلا', 'وما', 'ليست', 'لست', 'لسنا', 'لستم', 'لستن', 'لسن', 'ليسا', 'ليستا', 'ليسوا', 'غير']
لا احب الكلام غير سبب


## 5. Clean all three sets

**What:** applies `clean_text` and then `remove_stopwords` to the `TWEET` column of train, validation and test, and stores the result in a new column `CLEAN` (the original `TWEET` stays untouched). It shows 8 random before/after examples from train and counts how many tweets became empty.

**Why:** all three sets must be cleaned with exactly the same function. The 8 examples go into the report. We **never delete rows** (the split is final); an empty tweet simply becomes a sequence of only zeros after padding, so we just report how many there are.

In [10]:
def preprocess(text):
    return remove_stopwords(clean_text(text))

for name, d in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    d["CLEAN"] = d["TWEET"].fillna("").map(preprocess)

# 8 before/after examples for the report (fixed seed -> same examples every run)
examples_df = train_df.sample(8, random_state=SEED)[["ID", "TWEET", "CLEAN", "LABEL"]].reset_index(drop=True)
pd.set_option("display.max_colwidth", None)
display(examples_df)

# Empty tweets after cleaning (kept, only reported)
for name, d in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    n_empty = int((d["CLEAN"] == "").sum())
    print(f"{name:<10} empty after cleaning: {n_empty}")
    add_stat(f"empty_after_cleaning_{name}", n_empty)

,ID,TWEET,CLEAN,LABEL
0,1121,للحين ماطلعت نتائج القدرات ياربي خايفه ....,للحين ماطلعت نتائج القدرات ياربي خايفه,fear
1,10013,اجمل ماقيل بالحب عن الام امي لا تؤم القلوب الا اليك ولا تلين الصخور الا لحنانك انتِ و الجنه تحت قدميك احبك امي,اجمل ماقيل بالحب الام امي لا تؤم القلوب ولا تلين الصخور لحنانك الجنه قدميك احبك امي,love
2,513,لم تفهمنـــــي .. ولـــــن تفهمنـــــــــي ­،،، فانــــا احدثـــك بقلبــــي .. وانــــت تحدثنـــــي بعقــــــلك!!!,لم تفهمني ولن تفهمني فانا احدثك بقلبي وانت تحدثني بعقلك,sadness
3,9646,لا تجعل كبريائك يقف في طريق عودتك لمن يحمل لكَ اكوامًا من الحُب❤️,لا تجعل كبريائك يقف طريق عودتك لمن يحمل اكواما الحب,love
4,3741,انتم اخر من يتكلم عن ال البيت ، ولايشرفني ان اناقش ناصبي تكفيري ارهابي لاعلاقه له بدين محمد اللهم العن السقيفه واصحابهاواتباعها,اخر يتكلم البيت ولايشرفني اناقش ناصبي تكفيري ارهابي لاعلاقه بدين محمد اللهم العن السقيفه واصحابهاواتباعها,anger
5,2489,تقرير عمرو اديب عن الموصل تقرير مرعب مرعب ربنا يحفظ بلدنا,تقرير عمرو اديب الموصل تقرير مرعب مرعب ربنا يحفظ بلدنا,fear
6,8195,الحمدلله علي الافلام والاغنيات والقهوه والكتب. تنجيك من الواقع.,الحمدلله علي الافلام والاغنيات والقهوه والكتب تنجيك الواقع,joy
7,9701,كنت زمان بستني سؤالك عني دلوقتي مبقاش يفرق.,كنت زمان بستني سؤالك عني دلوقتي مبقاش يفرق,sadness


train      empty after cleaning: 4
validation empty after cleaning: 1
test       empty after cleaning: 0


## 6. Fit the Keras Tokenizer (train only)

**What:** builds the vocabulary with `Tokenizer(num_words=20000)` using **only the training tweets**. It prints how many different words appear in train and computes `EMBED_INPUT_DIM`, the size of the embedding layer.

**Why:** fitting on train only prevents *data leakage*: the model must not learn anything from validation/test before evaluation. `num_words=20000` comes from the paper (2,000,000 embedding parameters ÷ 100 dimensions). Index 0 is reserved for padding, so the embedding needs `len(word_index) + 1` rows, but never more than 20,000. We use no `oov_token` (Keras default) because the paper does not mention one; unknown words are simply dropped.

In [11]:
tokenizer = Tokenizer(num_words=NUM_WORDS)          # no oov_token (Keras default)
tokenizer.fit_on_texts(train_df["CLEAN"])

vocab_size = len(tokenizer.word_index)
EMBED_INPUT_DIM = min(NUM_WORDS, vocab_size + 1)

print("Unique words in train (len(word_index)):", vocab_size)
print("EMBED_INPUT_DIM = min(20000, len + 1)   :", EMBED_INPUT_DIM)
add_stat("train_unique_words", vocab_size)
add_stat("embed_input_dim", EMBED_INPUT_DIM)

Unique words in train (len(word_index)): 20502
EMBED_INPUT_DIM = min(20000, len + 1)   : 20000


## 7. Convert to sequences and pad to length 64

**What:** turns every cleaned tweet into a list of word indices (`texts_to_sequences`) and then makes all lists exactly 64 long with `pad_sequences` (zeros added at the start, `padding="pre"`; extra words cut from the start, `truncating="pre"`). Before padding it prints length statistics and how many tweets were longer than 64.

**Why:** a neural network needs inputs of the same size. 64 is the paper's value. The statistics prove that 64 is long enough for our tweets: if no sequence is truncated, we lose no words. `pre` is the Keras default; the paper does not say which side.

In [12]:
seq_train = tokenizer.texts_to_sequences(train_df["CLEAN"])
seq_val   = tokenizer.texts_to_sequences(val_df["CLEAN"])
seq_test  = tokenizer.texts_to_sequences(test_df["CLEAN"])

for name, seqs in [("train", seq_train), ("validation", seq_val), ("test", seq_test)]:
    lens = np.array([len(s) for s in seqs])
    n_trunc = int((lens > MAX_LEN).sum())
    print(f"{name:<10} max={lens.max():3d}  mean={lens.mean():5.2f}  p95={np.percentile(lens, 95):5.1f}  truncated={n_trunc}")
    add_stat(f"seq_len_max_{name}", int(lens.max()))
    add_stat(f"seq_len_mean_{name}", round(float(lens.mean()), 2))
    add_stat(f"seq_len_p95_{name}", round(float(np.percentile(lens, 95)), 2))
    add_stat(f"truncated_{name}", n_trunc)

pad = dict(maxlen=MAX_LEN, padding=PADDING, truncating=TRUNCATING)
X_train = pad_sequences(seq_train, **pad)
X_val   = pad_sequences(seq_val, **pad)
X_test  = pad_sequences(seq_test, **pad)
print("\nExample padded sequence (train[0]):\n", X_train[0])

train      max= 27  mean=11.22  p95= 20.0  truncated=0
validation max= 22  mean= 7.77  p95= 15.0  truncated=0
test       max= 21  mean= 7.92  p95= 15.0  truncated=0

Example padded sequence (train[0]):
 [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0   13    2   21 3328 3329]


## 8. Out-of-vocabulary (OOV) statistics

**What:** counts what share of the words in validation and test are **not** in the tokenizer's top 20,000 words (so they are dropped), and what share of train words fall outside the top 20,000.

**Why:** these words are invisible to the model. A high OOV rate on validation/test explains part of the gap between training and test accuracy, and it is useful for the report and for Kordy's embedding work. Keras keeps only indices `< num_words`, so "top 20,000" means index 1 … 19,999.

In [13]:
word_index = tokenizer.word_index

def oov_share(texts):
    tokens = [w for t in texts for w in t.split()]
    oov = sum(1 for w in tokens if word_index.get(w, NUM_WORDS) >= NUM_WORDS)
    return oov, len(tokens), (oov / len(tokens) if tokens else 0.0)

for name, d in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    oov, total, share = oov_share(d["CLEAN"])
    print(f"{name:<10} tokens={total:6d}  outside top-20k={oov:5d}  share={share:.2%}")
    add_stat(f"tokens_{name}", total)
    add_stat(f"oov_tokens_{name}", oov)
    add_stat(f"oov_share_{name}", round(share, 4))

train      tokens= 50227  outside top-20k=  503  share=1.00%
validation tokens= 10505  outside top-20k= 3120  share=29.70%
test       tokens= 10685  outside top-20k= 3165  share=29.62%


## 9. Encode the labels

**What:** converts the text labels into numbers with the fixed class order: anger=0, joy=1, sadness=2, love=3, fear=4.

**Why:** the model outputs 5 numbers, one per class, so the labels must be integers. The order is fixed for the whole team so that "class 2" always means sadness in every notebook. The `assert` catches any misspelled or unexpected label.

In [14]:
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
print(LABEL2ID)

def encode(d):
    y = d["LABEL"].astype(str).str.strip().str.lower().map(LABEL2ID)
    assert y.notna().all(), f"unknown labels: {d.loc[y.isna(), 'LABEL'].unique()}"
    return y.to_numpy(dtype=np.int64)

y_train, y_val, y_test = encode(train_df), encode(val_df), encode(test_df)
print("train class counts:", np.bincount(y_train, minlength=5))

{'anger': 0, 'joy': 1, 'sadness': 2, 'love': 3, 'fear': 4}
train class counts: [1007  892  873  832  827]


## 10. Final checks

**What:** a group of `assert` statements that check the final shapes, that X and y have the same number of rows and that nothing is missing (NaN). It also reloads the tokenizer from JSON and checks it gives identical sequences.

**Why:** these checks guarantee the files we hand to the rest of the team are correct. If anything is wrong, the notebook stops here with an error instead of silently saving bad data.

In [15]:
assert X_train.shape == (4431, MAX_LEN), X_train.shape
assert X_val.shape == (950, MAX_LEN), X_val.shape
assert X_test.shape == (950, MAX_LEN), X_test.shape
assert len(y_train) == len(X_train) and len(y_val) == len(X_val) and len(y_test) == len(X_test)
for arr in (X_train, X_val, X_test, y_train, y_val, y_test):
    assert not np.isnan(arr.astype(float)).any()
for d in (train_df, val_df, test_df):
    assert d["CLEAN"].notna().all()
assert X_train.max() < EMBED_INPUT_DIM            # every index fits in the embedding

# The saved tokenizer must reproduce the same sequences (important for the demo)
tok_reloaded = tokenizer_from_json(tokenizer.to_json())
assert tok_reloaded.texts_to_sequences(val_df["CLEAN"]) == seq_val

print("All checks passed.")

All checks passed.


## 11. Save the outputs and download them

**What:** saves everything into an `outputs/` folder, zips it into `preprocessing_outputs.zip` and downloads it (only in Colab):

```
outputs/data/train_clean.csv, validation_clean.csv, test_clean.csv   (ID, TWEET, CLEAN, LABEL)
outputs/data/sequences.npz            (X_train, X_val, X_test, y_train, y_val, y_test)
outputs/data/tokenizer.json           (full Keras tokenizer)
outputs/data/word_index_top20k.json   (words with index < 20000)
outputs/data/preprocessing_config.json
outputs/results/preprocessing_stats.csv
outputs/results/preprocessing_examples.csv
```

**Why:** the other members need these files to build the embeddings and train the models without re-running preprocessing. We use `.npz` (one compressed file with all arrays) and not `.npy`, because our `.gitignore` ignores `*.npy`. The config file records every setting and library version so the results can be reproduced.

In [16]:
DATA_DIR, RES_DIR = "outputs/data", "outputs/results"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

cols = ["ID", "TWEET", "CLEAN", "LABEL"]
train_df[cols].to_csv(f"{DATA_DIR}/train_clean.csv", index=False, encoding="utf-8")
val_df[cols].to_csv(f"{DATA_DIR}/validation_clean.csv", index=False, encoding="utf-8")
test_df[cols].to_csv(f"{DATA_DIR}/test_clean.csv", index=False, encoding="utf-8")

np.savez(f"{DATA_DIR}/sequences.npz", X_train=X_train, X_val=X_val, X_test=X_test,
         y_train=y_train, y_val=y_val, y_test=y_test)

with open(f"{DATA_DIR}/tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tokenizer.to_json())

top20k = {w: i for w, i in word_index.items() if i < NUM_WORDS}
with open(f"{DATA_DIR}/word_index_top20k.json", "w", encoding="utf-8") as f:
    json.dump(top20k, f, ensure_ascii=False)

config = {
    "max_len": MAX_LEN, "num_words": NUM_WORDS, "embed_input_dim": EMBED_INPUT_DIM,
    "padding": PADDING, "truncating": TRUNCATING, "oov_token": None,
    "class_order": CLASSES, "label2id": LABEL2ID,
    "kept_negation_words": NEGATIONS, "stopword_count": len(STOP_WORDS),
    "tokenizer_fitted_on": "train only",
    "versions": {"python": sys.version.split()[0], "tensorflow": tf.__version__,
                 "keras": getattr(tf.keras, "__version__", "n/a"), "tokenizer_source": TOKENIZER_SOURCE,
                 "numpy": np.__version__, "pandas": pd.__version__,
                 "nltk": nltk.__version__, "emoji": emoji.__version__},
}
with open(f"{DATA_DIR}/preprocessing_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

pd.DataFrame(stats).to_csv(f"{RES_DIR}/preprocessing_stats.csv", index=False, encoding="utf-8")
examples_df.to_csv(f"{RES_DIR}/preprocessing_examples.csv", index=False, encoding="utf-8")

# Zip the folder (the zip contains outputs/...) and download in Colab
shutil.make_archive("preprocessing_outputs", "zip", root_dir=".", base_dir="outputs")
print("Saved files:")
for root, _, fs in os.walk("outputs"):
    for fn in sorted(fs):
        print(" ", os.path.join(root, fn))

if "google.colab" in sys.modules:
    from google.colab import files
    files.download("preprocessing_outputs.zip")
else:
    print("Not running in Colab: download skipped (zip is in the current folder).")

Saved files:
  outputs/results/preprocessing_examples.csv
  outputs/results/preprocessing_stats.csv
  outputs/data/preprocessing_config.json
  outputs/data/sequences.npz
  outputs/data/test_clean.csv
  outputs/data/tokenizer.json
  outputs/data/train_clean.csv
  outputs/data/validation_clean.csv
  outputs/data/word_index_top20k.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Hand-off notes for the team

- **Kordy (embeddings):** use `outputs/data/word_index_top20k.json` (word → index, only indices < 20,000) to build the embedding matrix row by row, `preprocessing_config.json` for `embed_input_dim` (number of rows) and `max_len`, and the `CLEAN` column of the `*_clean.csv` files if you train or adapt word vectors. Row 0 of the matrix is padding and should stay zeros.
- **Yassin and Mahmoud (models):** load `outputs/data/sequences.npz` with `np.load(...)` → `X_train, X_val, X_test, y_train, y_val, y_test`. Use `input_dim = embed_input_dim` and `input_length = 64` in the Embedding layer. Labels are integers 0–4 in the order anger, joy, sadness, love, fear (use `sparse_categorical_crossentropy`, or one-hot them for `categorical_crossentropy`).
- **Malak (demo):** a new tweet must go through **exactly** the same pipeline: the same `clean_text` function, the same stop-word list (NLTK Arabic, cleaned, without لا، لم، لن، ما، ليس), then `tokenizer_from_json(open("tokenizer.json").read())`, `texts_to_sequences`, and `pad_sequences(maxlen=64, padding="pre", truncating="pre")`. Any difference will silently lower accuracy.